# Creating linear model to predict value of rating

In [ ]:
import os
import re
import ast
import math
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from underthesea import word_tokenize, chunk
from gensim.models import Word2Vec


# ------------------------
# Global config and state
# ------------------------
DATASET_FILENAME = "foody_combined_data_final.csv"
STOPWORDS_FILENAME = "stopwords.txt"
META_PREPROCESSED_FILENAME = "preprocessed_store_infos.csv"
META_FILENAME = "store_infos.csv"
W2V_VECTOR_SIZE = 200
TIME_MARGIN_MINUTES = 180

DAY_COLS = ["Thứ hai", "Thứ ba", "Thứ tư", "Thứ năm", "Thứ sáu", "Thứ bảy", "Ngày nghỉ"]
DAY_ALIASES = {
    0: ["thu hai", "thu 2"],
    1: ["thu ba", "thu 3"],
    2: ["thu tu", "thu 4"],
    3: ["thu nam", "thu 5"],
    4: ["thu sau", "thu 6"],
    5: ["thu bay", "thu 7"],
    6: ["chu nhat"],
}
TIME_SLOT_DEFAULTS = {
    "sang": (7, 0, 11, 0),
    "trua": (11, 0, 13, 30),
    "chieu": (14, 0, 17, 30),
    "toi": (18, 0, 22, 30),
    "dem": (18, 0, 22, 30),
}
LOCATION_PREFIX_RE = r"^(?:quan|q|huyen|h|thi xa|tx|thanh pho|tp|phuong|p|xa)\s+"

SERVICE_SYNONYMS = {
    "giao_tan_noi": ["giao tận nơi", "giao hang", "giao hàng", "ship"],
    "dat_ban": ["đặt bàn", "dat ban", "booking", "đặt chỗ", "dat cho"],
    "khuyen_mai": ["khuyến mãi", "khuyen mai", "ưu đãi", "uu dai", "promotion"],
    "phuc_vu": ["phục vụ", "phuc vu", "nhân viên", "nhan vien", "service"],
}
SERVICE_VOCAB = {key: i for i, key in enumerate(SERVICE_SYNONYMS.keys())}

ASPECT_ALIAS = {
    "vi_tri": ["vi tri", "dia diem", "dia chi"],
    "gia_ca": ["gia ca", "gia", "re", "dat", "phai chang"],
    "chat_luong": ["chat luong", "ngon", "do an ngon"],
    "phuc_vu": ["phuc vu", "nhan vien", "service"],
    "khong_gian": ["khong gian", "view", "dep", "thoang"],
}
RATING_ORDER = ["vi_tri", "gia_ca", "chat_luong", "phuc_vu", "khong_gian"]

# runtime globals
_STORE_DF_FOR_PARSE: pd.DataFrame = pd.DataFrame()
_STOPWORDS_RAW: set[str] = set()
_STOPWORDS_CANON: set[str] = set()
_LOCATION_VOCAB: dict[str, int] = {}
_LOCATION_LABELS: dict[str, str] = {}
_LOCATION_ALIAS_TO_KEY: dict[str, str] = {}
_CUISINE_VOCAB: dict[str, int] = {}
_CUISINE_LABELS: dict[str, str] = {}
_TARGET_VOCAB: dict[str, int] = {}
_TARGET_LABELS: dict[str, str] = {}
_CATEGORY_VOCAB: dict[str, int] = {}
_CATEGORY_LABELS: dict[str, str] = {}
_META_CORPUS: list[list[str]] = []
_META_W2V: Word2Vec | None = None

store_df = None

## General preprocessing and initalize global variable

In [2]:
def _normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def _strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", str(text)) if unicodedata.category(c) != "Mn")


def _canon(text: str) -> str:
    return _strip_accents(_normalize_text(text)).lower()


def _query_tokens(user_query: str, drop_stopwords: bool = True) -> list[str]:
    text = re.sub(r"[^\w\s]", " ", _canon(user_query))
    tokens = [t for t in text.split() if t]
    if drop_stopwords:
        tokens = [t for t in tokens if t not in _STOPWORDS_CANON]
    return tokens


def _query_text(user_query: str, drop_stopwords: bool = True) -> str:
    return " ".join(_query_tokens(user_query, drop_stopwords=drop_stopwords))


def _split_pipe_values(raw_value) -> list[str]:
    if pd.isna(raw_value):
        return []
    return [x.strip() for x in re.findall(r"[^|]+", str(raw_value)) if x.strip()]


def _is_positive(raw_value) -> bool:
    if pd.isna(raw_value):
        return False
    if isinstance(raw_value, (int, float, np.integer, np.floating)):
        return float(raw_value) > 0
    text = _canon(raw_value)
    return text in {"1", "true", "yes", "co", "c"} or "co" in text


def _to_minutes(hour: int, minute: int) -> int:
    hour = max(0, min(23, int(hour)))
    minute = max(0, min(59, int(minute)))
    return hour * 60 + minute


def _minutes_to_cyc(minutes: int) -> np.ndarray:
    rad = 2 * math.pi * (float(minutes) / 1440.0)
    return np.array([math.sin(rad), math.cos(rad)], dtype=float)


def _find_existing_path(filename: str, include_model_folder: bool = False) -> Path | None:
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    candidates = []
    for root in roots:
        candidates.append(root / filename)
        candidates.append(root / "dataset" / filename)
        candidates.append(root / "tmp" / filename)
        if include_model_folder:
            candidates.append(root / "model" / "linear model" / filename)

    for p in candidates:
        rp = p.resolve()
        if rp.exists():
            return rp
    return None


def _load_store_df() -> pd.DataFrame:
    if "store_df" in globals() and isinstance(store_df, pd.DataFrame):
        return store_df.copy()
    path = _find_existing_path(DATASET_FILENAME)
    if path is None:
        raise FileNotFoundError(f"Cannot locate {DATASET_FILENAME}")
    return pd.read_csv(path)


def _load_stopwords() -> set[str]:
    if "stopwords" in globals() and isinstance(stopwords, set):
        return {str(x).strip() for x in stopwords if str(x).strip()}

    path = _find_existing_path(STOPWORDS_FILENAME, include_model_folder=True)
    if path is None:
        return set()

    with open(path, "r", encoding="utf-8") as f:
        stopwords = {x.strip() for x in f.read().split() if x.strip()}
        return stopwords

def _load_meta_corpus() -> list[list[str]]:
    if _META_CORPUS:
        return _META_CORPUS
    
    path = _find_existing_path(META_PREPROCESSED_FILENAME)
    if path is None:
        return []
    
    df = pd.read_csv(path, encoding="utf-8")
    col = "MetaKeywords" if "MetaKeywords" in df.columns else df.columns[0]
    corpus = []
    for raw in df[col].astype(str):
        if pd.isna(raw) or not str(raw).strip():
            corpus.append([])
            continue
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                tokens = [str(t).strip().lower() for t in parsed if str(t).strip()]
            else:
                tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw))
        except Exception:
            tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw))
        if tokens:
            corpus.append(tokens)
        else:
            corpus.append([])
    return corpus

def _ensure_preprocessed_tmp() -> list[list[str]]:
    if _META_CORPUS:
       
        return _META_CORPUS
    path = _find_existing_path(META_PREPROCESSED_FILENAME)
    if path is not None:
        df = pd.read_csv(path, encoding="utf-8")
        col = "MetaKeywords" if "MetaKeywords" in df.columns else df.columns[0]
        corpus = []
        for raw in df[col].astype(str):
            if pd.isna(raw) or not str(raw).strip():
                corpus.append([])
                continue
            try:
                parsed = ast.literal_eval(raw)
                if isinstance(parsed, list):
                    tokens = [str(t).strip().lower() for t in parsed if str(t).strip()]
                else:
                    tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw.lower()))
            except Exception:
                tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw.lower()))
            if tokens:
                corpus.append(tokens)
            else:
                corpus.append([])
        return corpus

    corpus = []
    for raw in _STORE_DF_FOR_PARSE["MetaKeywords"].fillna("").astype(str):
        tokens = word_tokenize(raw)
        tokens = [t for t in tokens if _canon(t.lower()) not in _STOPWORDS_CANON]
        corpus.append(tokens)

    pd.DataFrame({"MetaKeywords": [str(tokens) for tokens in corpus]}).to_csv(
        os.path.join("..","..", 'tmp', META_PREPROCESSED_FILENAME),
        index=False,
        encoding="utf-8",
    )
    return corpus

def _build_vocab(terms: list[str]) -> tuple[dict[str, int], dict[str, str]]:
    canon_to_raw = {}
    for term in terms:
        c = _canon(term)
        if c and c not in canon_to_raw:
            canon_to_raw[c] = str(term).strip()
    vocab = {c: i for i, c in enumerate(sorted(canon_to_raw.keys()))}
    return vocab, canon_to_raw


def _build_location_alias_map(location_vocab: dict[str, int]) -> dict[str, str]:
    alias_to_key = {}
    for key in sorted(location_vocab.keys(), key=len, reverse=True):
        alias_to_key.setdefault(key, key)
        stripped = re.sub(LOCATION_PREFIX_RE, "", key).strip()
        if stripped:
            alias_to_key.setdefault(stripped, key)
            alias_to_key.setdefault(f"quan {stripped}", key)
            alias_to_key.setdefault(f"huyen {stripped}", key)
    return alias_to_key


def _vector_from_terms(terms: list[str], vocab: dict[str, int]) -> np.ndarray:
    vec = np.zeros(len(vocab), dtype=float)
    for term in terms:
        c = _canon(term)
        if c in vocab:
            vec[vocab[c]] = 1.0
    return vec


def _extract_vocab_matches(user_query: str, vocab_keys: list[str], max_hits: int = 6) -> list[str]:
    query = _query_text(user_query, drop_stopwords=True)
    matches = []
    for key in sorted(vocab_keys, key=len, reverse=True):
        if key and re.search(r"(?<!\w)" + re.escape(key) + r"(?!\w)", query):
            matches.append(key)
            if len(matches) >= max_hits:
                break
    return matches


def _extract_location_matches(user_query: str, max_hits: int = 3) -> list[str]:
    query = _query_text(user_query, drop_stopwords=False)
    hits = []
    seen = set()
    for alias in sorted(_LOCATION_ALIAS_TO_KEY.keys(), key=len, reverse=True):
        if re.search(r"(?<!\w)" + re.escape(alias) + r"(?!\w)", query):
            key = _LOCATION_ALIAS_TO_KEY[alias]
            if key not in seen:
                seen.add(key)
                hits.append(key)
                if len(hits) >= max_hits:
                    break
    return hits

def _embed_meta_tokens(tokens: list[str]) -> np.ndarray:
    if _META_W2V is None or not tokens:
        return np.zeros(W2V_VECTOR_SIZE, dtype=float)
    vecs = []
    for token in tokens:
        t = _normalize_text(token)
        if t in _META_W2V.wv:
            vecs.append(_META_W2V.wv[t])
    if not vecs:
        return np.zeros(W2V_VECTOR_SIZE, dtype=float)
    return np.mean(np.asarray(vecs, dtype=float), axis=0)


def _init_runtime() -> None:
    global _STORE_DF_FOR_PARSE
    global _STOPWORDS_RAW, _STOPWORDS_CANON
    global _LOCATION_VOCAB, _LOCATION_LABELS, _LOCATION_ALIAS_TO_KEY
    global _CUISINE_VOCAB, _CUISINE_LABELS
    global _TARGET_VOCAB, _TARGET_LABELS
    global _CATEGORY_VOCAB, _CATEGORY_LABELS
    global _META_CORPUS, _META_W2V
    _STORE_DF_FOR_PARSE = _load_store_df()

    _STOPWORDS_RAW = _load_stopwords()
    _STOPWORDS_CANON = {_canon(x) for x in _STOPWORDS_RAW if _canon(x)}

    location_terms = []
    for col in ["District", "Area"]:
        if col in _STORE_DF_FOR_PARSE.columns:
            location_terms.extend(_STORE_DF_FOR_PARSE[col].dropna().astype(str).tolist())

    cuisine_terms = []
    if "Cuisines" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["Cuisines"].dropna().astype(str):
            cuisine_terms.extend(_split_pipe_values(raw))

    target_terms = []
    if "LstTargetAudience" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["LstTargetAudience"].dropna().astype(str):
            target_terms.extend(_split_pipe_values(raw))

    category_terms = []
    if "LstCategory" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["LstCategory"].dropna().astype(str):
            category_terms.extend(_split_pipe_values(raw))

    _LOCATION_VOCAB, _LOCATION_LABELS = _build_vocab(location_terms)
    _LOCATION_ALIAS_TO_KEY = _build_location_alias_map(_LOCATION_VOCAB)
    _CUISINE_VOCAB, _CUISINE_LABELS = _build_vocab(cuisine_terms)
    _TARGET_VOCAB, _TARGET_LABELS = _build_vocab(target_terms)
    _CATEGORY_VOCAB, _CATEGORY_LABELS = _build_vocab(category_terms)

    _META_CORPUS = _ensure_preprocessed_tmp() or [["empty"]]
    if _META_CORPUS != [["empty"]] and _META_W2V is None:
        _META_W2V = Word2Vec(
            sentences=_META_CORPUS,
            vector_size=W2V_VECTOR_SIZE,
            window=5,
            min_count=1,
            workers=2,
            seed=42,
        )
    
_init_runtime()

## Parsing and embeding functions
### Store information parsing and embedding

In [3]:
def _parse_open_close_from_text(raw_value) -> tuple[int, int] | None:
    if pd.isna(raw_value):
        return None
    text = _canon(raw_value)

    hhmm = re.findall(r"(\d{1,2}):(\d{1,2})", text)
    if len(hhmm) >= 2:
        return (_to_minutes(int(hhmm[0][0]), int(hhmm[0][1])), _to_minutes(int(hhmm[1][0]), int(hhmm[1][1])))

    compact = re.findall(r"(\d{1,2})h(\d{0,2})", text)
    if len(compact) >= 2:
        m1 = int(compact[0][1]) if compact[0][1] else 0
        m2 = int(compact[1][1]) if compact[1][1] else 0
        return (_to_minutes(int(compact[0][0]), m1), _to_minutes(int(compact[1][0]), m2))

    return None

def _collect_store_schedules(store_row: pd.Series) -> dict[int, tuple[int, int]]:
    schedules = {}
    for idx, day_col in enumerate(DAY_COLS):
        if day_col in store_row.index:
            open_close = _parse_open_close_from_text(store_row[day_col])
            if open_close is not None:
                schedules[idx] = open_close
    return schedules

def build_store_location_embedding(store_row: pd.Series) -> np.ndarray:
    terms = [str(store_row[col]) for col in ["District", "Area"] if col in store_row.index and pd.notna(store_row[col])]
    return _vector_from_terms(terms, _LOCATION_VOCAB)


def build_store_cuisine_embedding(store_row: pd.Series) -> np.ndarray:
    terms = [str(store_row["Cuisines"])] if "Cuisines" in store_row.index and pd.notna(store_row["Cuisines"]) else []
    return _vector_from_terms(terms, _CUISINE_VOCAB)


def build_store_target_audience_embedding(store_row: pd.Series) -> np.ndarray:
    terms = _split_pipe_values(store_row["LstTargetAudience"]) if "LstTargetAudience" in store_row.index and pd.notna(store_row["LstTargetAudience"]) else []
    return _vector_from_terms(terms, _TARGET_VOCAB)


def build_store_category_embedding(store_row: pd.Series) -> np.ndarray:
    terms = _split_pipe_values(store_row["LstCategory"]) if "LstCategory" in store_row.index and pd.notna(store_row["LstCategory"]) else []
    return _vector_from_terms(terms, _CATEGORY_VOCAB)


def build_store_service_embedding(store_row: pd.Series) -> np.ndarray:
    vec = np.zeros(len(SERVICE_VOCAB), dtype=float)
    if _is_positive(store_row.get("HasDelivery", np.nan)) or _is_positive(store_row.get("Giao tận nơi", np.nan)):
        vec[SERVICE_VOCAB["giao_tan_noi"]] = 1.0
    if _is_positive(store_row.get("HasBooking", np.nan)) or _is_positive(store_row.get("Đặt bàn", np.nan)):
        vec[SERVICE_VOCAB["dat_ban"]] = 1.0
    if _is_positive(store_row.get("HasPromotion", np.nan)):
        vec[SERVICE_VOCAB["khuyen_mai"]] = 1.0

    service_text = " ".join(
        str(store_row[c])
        for c in ["AccessGuide", "MetaKeywords"]
        if c in store_row.index and pd.notna(store_row[c])
    )
    service_canon = _canon(service_text)
    for key, aliases in SERVICE_SYNONYMS.items():
        if any(_canon(alias) in service_canon for alias in aliases):
            vec[SERVICE_VOCAB[key]] = 1.0
    return vec


def build_store_rating_embedding(store_row: pd.Series) -> np.ndarray:
    aspect_cols = {
        "vi_tri": ["Vị trí", "Vị Trí"],
        "gia_ca": ["Giá cả", "Giá Cả"],
        "chat_luong": ["Chất lượng", "Chất Lượng"],
        "phuc_vu": ["Phục vụ", "Phục Vụ"],
        "khong_gian": ["Không gian", "Không Gian"],
    }

    def _pick_float(candidates: list[str]) -> float:
        for c in candidates:
            if c in store_row.index and pd.notna(store_row[c]):
                try:
                    return float(store_row[c])
                except Exception:
                    pass
        return np.nan

    vals = [_pick_float(aspect_cols[k]) for k in RATING_ORDER]
    if np.isnan(vals).all():
        exc = float(store_row.get("Excellent", 0) or 0)
        good = float(store_row.get("Good", 0) or 0)
        avg = float(store_row.get("Average", 0) or 0)
        bad = float(store_row.get("Bad", 0) or 0)
        total = exc + good + avg + bad
        general = 0.0 if total == 0 else 10.0 * (exc + 0.75 * good + 0.5 * avg + 0.25 * bad) / total
        vals = [general] * 5

    arr = np.array([0.0 if np.isnan(v) else float(v) for v in vals], dtype=float)
    return np.concatenate([arr, np.array([float(arr.mean())], dtype=float)])


def build_store_time_embedding(store_row: pd.Series, preferred_day_index: int | None = None) -> np.ndarray:
    schedules = _collect_store_schedules(store_row)
    if preferred_day_index is not None and preferred_day_index in schedules:
        day_idx = preferred_day_index
    elif schedules:
        day_idx = sorted(schedules.keys())[0]
    else:
        day_idx = 0

    open_min, close_min = schedules.get(day_idx, (0, 0))
    day_vec = np.zeros(7, dtype=float)
    day_vec[day_idx] = 1.0
    return np.concatenate([_minutes_to_cyc(open_min), _minutes_to_cyc(close_min), day_vec])




### User parsing and embeding

In [4]:
def _price_token_to_vnd(number_text: str, unit_text: str | None) -> int:
    value = float(str(number_text).replace(".", "").replace(",", "."))
    unit = _canon(unit_text or "")
    if unit in {"k", "nghin", "ngan"}:
        value *= 1000
    elif unit in {"tr", "trieu"}:
        value *= 1_000_000
    return int(value)

def parse_user_location(user_query: str) -> dict:
    hits = _extract_location_matches(user_query, max_hits=3)
    return {
        "value": [_LOCATION_LABELS[h] for h in hits if h in _LOCATION_LABELS],
        "embedding": _vector_from_terms(hits, _LOCATION_VOCAB),
    }


def parse_user_price(user_query: str) -> tuple[float, float]:
    text_canon = _query_text(user_query, drop_stopwords=False)
    pattern = r"(\d+(?:[\.,]\d+)?)\s*(k|nghin|ngan|trieu|tr|vnd|d|dong)?"
    values = []

    for match in re.finditer(pattern, text_canon):
        num, unit = match.group(1), match.group(2)
        start, end = match.span()
        left_context = text_canon[max(0, start - 12):start]
        local_context = text_canon[max(0, start - 12):min(len(text_canon), end + 12)]

        if re.search(r"(quan|thu)\s*$", left_context):
            continue

        try:
            value_vnd = _price_token_to_vnd(num, unit)
        except Exception:
            continue

        if unit is None:
            if value_vnd < 1000:
                continue
            if not re.search(r"gia|price|duoi|tren|toi da|toi thieu|khoang|tam|tu", local_context):
                continue

        values.append(value_vnd)

    if not values:
        return (0.0, float("inf"))
    if len(values) >= 2:
        return (float(min(values[0], values[1])), float(max(values[0], values[1])))

    single = float(values[0])
    if re.search(r"(tu|tren|it nhat|toi thieu)", text_canon):
        return (single, float("inf"))
    return (0.0, single)


def parse_user_cuisine(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_CUISINE_VOCAB.keys()), max_hits=6)
    return {
        "value": [_CUISINE_LABELS[h] for h in hits if h in _CUISINE_LABELS],
        "embedding": _vector_from_terms(hits, _CUISINE_VOCAB),
    }


def parse_user_target_audience(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_TARGET_VOCAB.keys()), max_hits=6)
    return {
        "value": [_TARGET_LABELS[h] for h in hits if h in _TARGET_LABELS],
        "embedding": _vector_from_terms(hits, _TARGET_VOCAB),
    }


def parse_user_categories(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_CATEGORY_VOCAB.keys()), max_hits=6)
    return {
        "value": [_CATEGORY_LABELS[h] for h in hits if h in _CATEGORY_LABELS],
        "embedding": _vector_from_terms(hits, _CATEGORY_VOCAB),
    }


def parse_user_rating(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=False)
    requested = [aspect for aspect, aliases in ASPECT_ALIAS.items() if any(a in text for a in aliases)]

    desired_score = None
    for pattern in [
        r"(\d+(?:[\.,]\d+)?)\s*/\s*10",
        r"(\d+(?:[\.,]\d+)?)\s*(?:diem|sao|star)",
        r"danh gia\s*(\d+(?:[\.,]\d+)?)",
    ]:
        m = re.search(pattern, text)
        if m:
            score = float(m.group(1).replace(",", "."))
            if 0 <= score <= 10:
                desired_score = score
                break

    if desired_score is None:
        if any(w in text for w in ["excellent", "xuat sac", "rat tot", "tuyet voi"]):
            desired_score = 8.0
        elif any(w in text for w in ["tot", "dep", "on", "ok"]):
            desired_score = 6.5
        elif any(w in text for w in ["trung binh", "average"]):
            desired_score = 5.0
        elif any(w in text for w in ["te", "xau", "bad", "kem"]):
            desired_score = 2.0
        else:
            desired_score = 6.5

    aspect_vec = np.zeros(5, dtype=float)
    if requested:
        for i, aspect in enumerate(RATING_ORDER):
            aspect_vec[i] = desired_score if aspect in requested else 1.0
    else:
        aspect_vec[:] = desired_score

    emb = np.concatenate([aspect_vec, np.array([desired_score], dtype=float)])
    return {"requested_aspects": requested, "desired_score": desired_score, "embedding": emb}


def parse_user_time(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=False)

    day_index = None
    for idx, aliases in DAY_ALIASES.items():
        if any(a in text for a in aliases):
            day_index = idx
            break

    open_min = None
    close_min = None

    am_pm = re.search(r"(\d{1,2})(?::(\d{1,2}))?\s*(am|pm)", text)
    if am_pm:
        hour = int(am_pm.group(1))
        minute = int(am_pm.group(2)) if am_pm.group(2) else 0
        if am_pm.group(3) == "pm" and hour < 12:
            hour += 12
        if am_pm.group(3) == "am" and hour == 12:
            hour = 0
        open_min = _to_minutes(hour, minute)
        close_min = open_min

    if open_min is None:
        range_match = re.search(r"(\d{1,2})(?:[:hg](\d{1,2}))?\s*(?:-|den|toi|to)\s*(\d{1,2})(?:[:hg](\d{1,2}))?", text)
        if range_match:
            h1 = int(range_match.group(1))
            m1 = int(range_match.group(2)) if range_match.group(2) else 0
            h2 = int(range_match.group(3))
            m2 = int(range_match.group(4)) if range_match.group(4) else 0
            open_min = _to_minutes(h1, m1)
            close_min = _to_minutes(h2, m2)
        else:
            single_match = re.search(r"(\d{1,2})(?:[:hg](\d{1,2}))?", text)
            if single_match:
                h = int(single_match.group(1))
                m = int(single_match.group(2)) if single_match.group(2) else 0
                open_min = _to_minutes(h, m)
                close_min = open_min

    if any(x in text for x in ["chieu", "toi", "dem"]):
        if open_min is not None and open_min < 12 * 60:
            open_min += 12 * 60
        if close_min is not None and close_min < 12 * 60:
            close_min += 12 * 60
    elif "sang" in text:
        if open_min == 12 * 60:
            open_min = 0
        if close_min == 12 * 60:
            close_min = 0

    if open_min is None or close_min is None:
        slot = next((name for name in TIME_SLOT_DEFAULTS if name in text), None)
        if slot is None:
            open_min, close_min = _to_minutes(0, 0), _to_minutes(23, 59)
        else:
            h1, m1, h2, m2 = TIME_SLOT_DEFAULTS[slot]
            open_min, close_min = _to_minutes(h1, m1), _to_minutes(h2, m2)

    if day_index is None:
        day_vec = np.full(7, 1.0 / 7.0, dtype=float)
    else:
        day_vec = np.zeros(7, dtype=float)
        day_vec[day_index] = 1.0

    emb = np.concatenate([_minutes_to_cyc(open_min), _minutes_to_cyc(close_min), day_vec])
    return {"day_index": day_index, "opening_minute": int(open_min), "closing_minute": int(close_min), "embedding": emb}


def parse_user_service(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=True)
    labels = [key for key, aliases in SERVICE_SYNONYMS.items() if any(_canon(alias) in text for alias in aliases)]
    vec = np.zeros(len(SERVICE_VOCAB), dtype=float)
    for key in labels:
        vec[SERVICE_VOCAB[key]] = 1.0
    return {"value": labels, "embedding": vec}


def parse_user_misc(user_query: str) -> dict:
    tokens = _query_tokens(user_query, drop_stopwords=True)
    return {"value": tokens, "embedding": _embed_meta_tokens(tokens)}


def parse_user_all(user_query: str) -> dict:
    return {
        "location": parse_user_location(user_query),
        "price": parse_user_price(user_query),
        "cuisine": parse_user_cuisine(user_query),
        "target_audience": parse_user_target_audience(user_query),
        "categories": parse_user_categories(user_query),
        "rating": parse_user_rating(user_query),
        "time": parse_user_time(user_query),
        "service": parse_user_service(user_query),
        "misc": parse_user_misc(user_query),
    }

## Similarity measuremets Functions

In [5]:
# Embedding-based time-fit overrides

def compare_embeddings(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    if vec_a.shape != vec_b.shape:
        return 0.0
    denom = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)

def _day_distance(day_a: int, day_b: int) -> int:
    diff = abs(int(day_a) - int(day_b))
    return min(diff, 7 - diff)


def _directional_margin_score(delta_good: int, cap: int = TIME_MARGIN_MINUTES) -> float:
    cap = max(1, int(cap))
    if delta_good >= 0:
        return min(1.0, 0.7 + 0.3 * (delta_good / cap))
    return max(0.0, 0.7 + (delta_good / cap))

def _cyc_pair_to_minutes(cyc_vec: np.ndarray) -> int:
    """Convert [sin, cos] cyclic pair back to minute-of-day."""
    if cyc_vec is None or len(cyc_vec) != 2:
        return 0
    angle = float(np.arctan2(float(cyc_vec[0]), float(cyc_vec[1])))
    if angle < 0:
        angle += 2 * np.pi
    return int(round(angle * 1440.0 / (2 * np.pi))) % 1440


def _time_fit_from_embeddings(
    user_embedding: np.ndarray,
    store_embedding: np.ndarray,
    margin_minutes: int = TIME_MARGIN_MINUTES,
) -> float:
    """Score time fitness in [0,1] from user/store time embeddings."""
    if user_embedding.shape != store_embedding.shape or user_embedding.size < 11:
        return 0.0

    base_sim = max(0.0, compare_embeddings(user_embedding, store_embedding))

    user_open = _cyc_pair_to_minutes(user_embedding[0:2])
    user_close = _cyc_pair_to_minutes(user_embedding[2:4])
    store_open = _cyc_pair_to_minutes(store_embedding[0:2])
    store_close = _cyc_pair_to_minutes(store_embedding[2:4])

    open_fit = _directional_margin_score(user_open - store_open, cap=margin_minutes)
    close_fit = _directional_margin_score(store_close - user_close, cap=margin_minutes)
    

    score = 0.45 * base_sim + 0.25 * open_fit + 0.25 * close_fit
    store_days = store_embedding[-7:].tolist()
    user_days:list = user_embedding[-7:].tolist()
    if user_days.count(1) > 0 and store_days.count(1) > 0:
        user_day = user_days.index(1)
        store_day = store_days.index(1)
        error_day = _day_distance(user_day, store_day)
        score -= 0.15 * error_day
    
    return float(np.clip(score, -1.0, 1.0))

def all_time_embeding_compare( user_embedding: np.ndarray,
    store_embeddings: list[np.ndarray], margin_minutes: int = TIME_MARGIN_MINUTES,) -> float:
    """Compare user time embedding with multiple store time embeddings, return best score."""
    
    best_score = 0.0
    for day_idx in range(7):
        store_embedding = store_embeddings[day_idx] if day_idx < len(store_embeddings) else None
        if store_embedding is None:
            continue
        score = _time_fit_from_embeddings(
            user_embedding=user_embedding,
            store_embedding=store_embedding,
            margin_minutes=margin_minutes,
        )
        best_score = max(best_score, score)

    return float(np.clip(best_score, -1.0, 1.0))

def compare_store_time_fit(
    user_time: dict,
    store_row: pd.Series,
    margin_minutes: int = TIME_MARGIN_MINUTES,
) -> float:
    """Compare user/store time using embeddings from parse_user_time and build_store_time_embedding."""
    
    if not isinstance(user_time, dict):
        return 0.0

    user_embedding = np.asarray(user_time.get("embedding", []), dtype=float)
    
    if user_embedding.size == 0:
        return 0.0
    
    schedules = _collect_store_schedules(store_row)
    if not schedules:
        return 0.0

    req_day = user_time.get("day_index", None)
    if req_day is not None and req_day in schedules:
        candidate_days = [int(req_day)]
    else:
        candidate_days = list(schedules.keys())

    best_score = 0.0
    for day_idx in candidate_days:
        store_embedding = np.asarray(
            build_store_time_embedding(store_row, preferred_day_index=day_idx),
            dtype=float,
        )
        score = _time_fit_from_embeddings(
            user_embedding=user_embedding,
            store_embedding=store_embedding,
            margin_minutes=margin_minutes,
        )
        best_score = max(best_score, score)

    return float(np.clip(best_score, -1.0, 1.0))


def compare_time_from_query(
    user_query: str,
    store_row: pd.Series,
    margin_minutes: int = TIME_MARGIN_MINUTES,
) -> float:
    """Parse query -> user time embedding, then compare with store time embedding."""
    
    user_time = parse_user_time(user_query)
    
    return compare_store_time_fit(
        user_time=user_time,
        store_row=store_row,
        margin_minutes=margin_minutes,
    )

In [ ]:
# Quick validation for time-fit scoring behavior and location extraction
user_time = parse_user_time("tôi cần một quán ăn tại 3 tối thứ 2")

score_open_2pm = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "14:00 - 21:00"}))
score_open_3pm = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "15:00 - 21:00"}))

score_close_15 = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "10:00 - 15:00"}))
score_close_17 = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "10:00 - 17:00"}))

score_same_day = compare_store_time_fit(user_time, pd.Series({"Thứ hai": "15:00 - 21:00"}))
score_diff_day = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "15:00 - 21:00"}))

location_check = parse_user_location("Tôi muốn tìm quán ăn ở khu vực Tân Bình")

print("open@2pm > open@3pm:", score_open_2pm, ">", score_open_3pm)
print("close@5pm > close@3pm:", score_close_17, ">", score_close_15)
print("same-day score > different-day score:", score_same_day, ">", score_diff_day)
print("location parse:", location_check["value"])

open@2pm > open@3pm: 0.14488887394336036 > 0.125
close@5pm > close@3pm: 0.193726667333044 > 0.16382285676537822
same-day score > different-day score: 0.7250000000000001 > 0.125
different-day score is negative: 0.125
location parse: ['Quận Tân Bình']


## Build / loading Artifacts

In [7]:
# Build or load tmp artifacts used by the main pipeline
TMP_DIR = Path.cwd().resolve().parents[1] / "tmp"
TMP_STORE_INFOS_PATH = TMP_DIR / "store_infos.csv"
TMP_PREPROCESSED_PATH = TMP_DIR / "preprocessed_store_infos.csv"
TMP_META_CORPUS_PATH = TMP_DIR / "meta_corpus.csv"
TMP_PARSED_STORE_EMB_PATH = TMP_DIR / "parsed_store_embeddings.csv"
TMP_DIR.mkdir(parents=True, exist_ok=True)


def _tokenize_meta_keywords(raw_value) -> list[str]:
    if pd.isna(raw_value):
        return []

    text = str(raw_value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            if parsed and isinstance(parsed[0], (tuple, list)):
                tokens = [str(item[0]).strip().lower() for item in parsed if len(item) > 0 and str(item[0]).strip()]
            else:
                tokens = [str(item).strip().lower() for item in parsed if str(item).strip()]
            if tokens:
                return tokens
    except Exception:
        pass

    return [tok for tok in re.findall(r"[\wÀ-ỹ]+", _normalize_text(text)) if tok]


def _ensure_store_infos_tmp() -> pd.DataFrame:
    if TMP_STORE_INFOS_PATH.exists():
        cached = pd.read_csv(TMP_STORE_INFOS_PATH, encoding="utf-8")
        if "MetaKeywords" in cached.columns:
            return cached

    base_df = _STORE_DF_FOR_PARSE if not _STORE_DF_FOR_PARSE.empty else _load_store_df()
    meta_col = base_df["MetaKeywords"] if "MetaKeywords" in base_df.columns else pd.Series([""] * len(base_df))
    out_df = pd.DataFrame({"MetaKeywords": meta_col.fillna("").astype(str)})
    out_df.to_csv(TMP_STORE_INFOS_PATH, index=False, encoding="utf-8")
    return out_df


def _parse_and_embed_store(store_row: pd.Series) -> dict:
    return {
        "location_emb": build_store_location_embedding(store_row).tolist(),
        "cuisine_emb": build_store_cuisine_embedding(store_row).tolist(),
        "target_audience_emb": build_store_target_audience_embedding(store_row).tolist(),
        "category_emb": build_store_category_embedding(store_row).tolist(),
        "service_emb": build_store_service_embedding(store_row).tolist(),
        "rating_emb": build_store_rating_embedding(store_row).tolist(),
        "time_emb": [
            build_store_time_embedding(store_row, preferred_day_index=day_idx).tolist()
            for day_idx in range(7)
        ],
    }

def build_or_load_store_embeddings_tmp(force_rebuild: bool = False) -> pd.DataFrame:
    global _META_CORPUS

    # store_infos_df = _ensure_store_infos_tmp()
    _META_CORPUS = _ensure_preprocessed_tmp()
    # _META_CORPUS = _ensure_meta_corpus_tmp(preprocessed_tokens)

    if TMP_PARSED_STORE_EMB_PATH.exists() and not force_rebuild:
        parsed_df = pd.read_csv(TMP_PARSED_STORE_EMB_PATH, encoding="utf-8")
        print(f"Loaded existing tmp file: {TMP_PARSED_STORE_EMB_PATH}")
        return parsed_df

    rows = []
    base_df = _STORE_DF_FOR_PARSE if not _STORE_DF_FOR_PARSE.empty else _load_store_df()
    for pos, (_, store_row) in enumerate(base_df.iterrows()):
        emb = _parse_and_embed_store(store_row)
        meta_tokens = _META_CORPUS[pos] if pos < len(_META_CORPUS) else []
        emb["MetaKeywords_emb"] = _embed_meta_tokens(meta_tokens).tolist()
        emb["store_id"] = int(store_row.get("RestaurantID", pos))
        rows.append(emb)

    parsed_df = pd.DataFrame(rows)
    parsed_df.to_csv(TMP_PARSED_STORE_EMB_PATH, index=False, encoding="utf-8")
    print(f"Created tmp file: {TMP_PARSED_STORE_EMB_PATH}")
    return parsed_df


    
parsed_dataset = build_or_load_store_embeddings_tmp(force_rebuild=False)
print("Parsed dataset preview:")
print(parsed_dataset.head())

Loaded existing tmp file: D:\python\ML project\Restaurant-recommendation-system\tmp\parsed_store_embeddings.csv
Parsed dataset preview:
                                        location_emb  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                                         cuisine_emb  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                        target_audience_emb  \
0  [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0]   
1  [1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0]   
2  [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0]   
3 

In [8]:
# Verify tmp artifacts used by the notebook pipeline
tmp_files = [
    TMP_PREPROCESSED_PATH,
    TMP_PARSED_STORE_EMB_PATH,
]

for p in tmp_files:
    print(f"{p.name}: {'exists' if p.exists() else 'missing'}")

preprocessed_store_infos.csv: exists
parsed_store_embeddings.csv: exists


## Creating linear model

### Dataset and query preprocessing

In [9]:


# ------------------------
# Data loading
# ------------------------
PROJECT_ROOT = Path.cwd().resolve().parents[1]
QUERY_RATING_PATH = PROJECT_ROOT / "dataset" / "restaurant_dataset_ver1.csv"
STORE_INFO_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "foody_combined_data_final (1).csv",
]
STORE_EMB_PATH = PROJECT_ROOT / "tmp" / "parsed_store_embeddings.csv"

store_info_path = next((p for p in STORE_INFO_CANDIDATES if p.exists()), None)
if store_info_path is None:
    raise FileNotFoundError("Cannot find foody_combined_data_final (1).csv")

# Build tmp embeddings lazily if missing
if not STORE_EMB_PATH.exists():
    _ = build_or_load_store_embeddings_tmp(force_rebuild=False)

query_rating_df = pd.read_csv(QUERY_RATING_PATH)
store_info_df = pd.read_csv(store_info_path)
store_emb_df = pd.read_csv(STORE_EMB_PATH)


def _parse_vector(raw_value) -> np.ndarray:
    if isinstance(raw_value, np.ndarray):
        return raw_value.astype(float).reshape(-1)
    if isinstance(raw_value, (list, tuple)):
        return np.asarray(raw_value, dtype=float).reshape(-1)
    if pd.isna(raw_value):
        return np.array([], dtype=float)

    text = str(raw_value).strip()
    if not text:
        return np.array([], dtype=float)

    try:
        parsed = ast.literal_eval(text)
        return np.asarray(parsed, dtype=float).reshape(-1)
    except Exception:
        pass

    cleaned = re.sub(r"\s+", " ", text.strip("[] "))
    if not cleaned:
        return np.array([], dtype=float)
    if "," in cleaned:
        parts = [p.strip() for p in cleaned.split(",") if p.strip()]
        return np.asarray([float(p) for p in parts], dtype=float).reshape(-1)
    return np.fromstring(cleaned, sep=" ", dtype=float).reshape(-1)


def _parse_time_embeddings(raw_value) -> list[np.ndarray]:
    if isinstance(raw_value, list):
        return [np.asarray(v, dtype=float).reshape(-1) for v in raw_value]
    if pd.isna(raw_value):
        return []

    text = str(raw_value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [np.asarray(v, dtype=float).reshape(-1) for v in parsed]
    except Exception:
        pass

    fallback = _parse_vector(raw_value)
    return [fallback] if fallback.size else []


EMBEDDING_COLS = [
    "location_emb",
    "cuisine_emb",
    "target_audience_emb",
    "category_emb",
    "service_emb",
    "rating_emb",
    "MetaKeywords_emb",
]
for col in EMBEDDING_COLS:
    if col in store_emb_df.columns:
        store_emb_df[col] = store_emb_df[col].apply(_parse_vector)
    else:
        store_emb_df[col] = [np.array([], dtype=float)] * len(store_emb_df)

if "time_emb" in store_emb_df.columns:
    store_emb_df["time_emb"] = store_emb_df["time_emb"].apply(_parse_time_embeddings)
else:
    store_emb_df["time_emb"] = [[] for _ in range(len(store_emb_df))]

store_emb_df["store_id"] = pd.to_numeric(store_emb_df["store_id"], errors="coerce")
store_emb_df = store_emb_df.dropna(subset=["store_id"]).copy()
store_emb_df["store_id"] = store_emb_df["store_id"].astype(int)

store_info_df["RestaurantID"] = pd.to_numeric(store_info_df["RestaurantID"], errors="coerce")
store_info_df = store_info_df.dropna(subset=["RestaurantID"]).copy()
store_info_df["RestaurantID"] = store_info_df["RestaurantID"].astype(int)

query_rating_df["restaurant_id"] = pd.to_numeric(query_rating_df["restaurant_id"], errors="coerce")
query_rating_df = query_rating_df.dropna(subset=["restaurant_id", "query", "label"]).copy()
query_rating_df["restaurant_id"] = query_rating_df["restaurant_id"].astype(int)
query_rating_df["label"] = pd.to_numeric(query_rating_df["label"], errors="coerce")
query_rating_df = query_rating_df.dropna(subset=["label"]).copy()

store_full_df = store_info_df.merge(
    store_emb_df,
    left_on="RestaurantID",
    right_on="store_id",
    how="inner",
)

train_pairs_df = query_rating_df.merge(
    store_full_df,
    left_on="restaurant_id",
    right_on="RestaurantID",
    how="inner",
)

if len(train_pairs_df) == 0:
    raise ValueError("No rows left after joining query/store/embedding tables")


def _safe_compare_embeddings(vec_a, vec_b) -> float:
    a = np.asarray(vec_a, dtype=float).reshape(-1)
    b = np.asarray(vec_b, dtype=float).reshape(-1)
    if a.size == 0 or b.size == 0:
        return 0.0
    return float(compare_embeddings(a, b))


def _price_compatibility(user_price: tuple[float, float], store_min, store_max) -> float:
    q_min, q_max = user_price
    if np.isinf(q_max) and q_min <= 0:
        return 0.0

    s_min = float(store_min) if not pd.isna(store_min) else 0.0
    s_max = float(store_max) if not pd.isna(store_max) else s_min
    if s_max < s_min:
        s_min, s_max = s_max, s_min

    q_min = float(q_min)
    q_max = 1e12 if np.isinf(q_max) else float(q_max)

    inter = max(0.0, min(s_max, q_max) - max(s_min, q_min))
    union = max(s_max, q_max) - min(s_min, q_min)
    if union <= 0:
        return 0.0
    return float(inter / union)


def _build_pair_features(parsed_query: dict, store_row: pd.Series) -> dict:
    return {
        "sim_location": _safe_compare_embeddings(parsed_query["location"]["embedding"], store_row["location_emb"]),
        "sim_cuisine": _safe_compare_embeddings(parsed_query["cuisine"]["embedding"], store_row["cuisine_emb"]),
        "sim_target_audience": _safe_compare_embeddings(parsed_query["target_audience"]["embedding"], store_row["target_audience_emb"]),
        "sim_category": _safe_compare_embeddings(parsed_query["categories"]["embedding"], store_row["category_emb"]),
        "sim_service": _safe_compare_embeddings(parsed_query["service"]["embedding"], store_row["service_emb"]),
        "sim_rating": _safe_compare_embeddings(parsed_query["rating"]["embedding"], store_row["rating_emb"]),
        "sim_meta": _safe_compare_embeddings(parsed_query["misc"]["embedding"], store_row["MetaKeywords_emb"]),
        "fit_time": float(all_time_embeding_compare(parsed_query["time"]["embedding"], store_row["time_emb"])),
        "fit_price": _price_compatibility(
            parsed_query["price"],
            store_row.get("PriceMin", np.nan),
            store_row.get("PriceMax", np.nan),
        ),
    }


# Cache parsed query embeddings so we parse each unique query only once.
query_parse_cache = {
    q: parse_user_all(str(q))
    for q in train_pairs_df["query"].dropna().astype(str).unique()
}

feature_rows = []
labels = []
groups = []
for _, row in train_pairs_df.iterrows():
    q = str(row["query"])
    parsed_query = query_parse_cache[q]
    feature_rows.append(_build_pair_features(parsed_query, row))
    labels.append(float(row["label"]))
    groups.append(q)

X_all = pd.DataFrame(feature_rows)
y_all = np.asarray(labels, dtype=float)
group_all = np.asarray(groups, dtype=str)

### Training and validating model using cross validation

In [13]:
# Model building libraries
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
# ------------------------
# Grouped 85/15 split
# ------------------------
TRAIN_PERCENT = 0.85
CV_TOP_K = 5
CV_RELEVANCE_THRESHOLD = 3.0
ALPHA_GRID = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]


def _split_queries_85_15(group_values: np.ndarray, seed: int = 42):
    unique_queries = pd.unique(group_values)
    rng = np.random.default_rng(seed)
    shuffled = np.array(unique_queries, dtype=object)
    rng.shuffle(shuffled)

    n = len(shuffled)
    if n < 2:
        raise ValueError("Need at least 2 unique queries for 85/15 split")

    n_train = max(1, int(np.floor(n * TRAIN_PERCENT)))
    if n_train >= n:
        n_train = n - 1

    train_queries = set(shuffled[:n_train])
    test_queries = set(shuffled[n_train:])
    return train_queries, test_queries


train_q, test_q = _split_queries_85_15(group_all, seed=42)

train_mask = np.array([g in train_q for g in group_all])
test_mask = np.array([g in test_q for g in group_all])

X_train, y_train, g_train = X_all.loc[train_mask], y_all[train_mask], group_all[train_mask]
X_test, y_test, g_test = X_all.loc[test_mask], y_all[test_mask], group_all[test_mask]

print(f"Rows: train={len(X_train)}, test={len(X_test)}")
print(f"Unique queries: train={len(set(g_train))}, test={len(set(g_test))}")


def _cv_dcg_at_k(relevances: list[float], k: int) -> float:
    rel = np.asarray(relevances[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rel.size + 2))
    gains = np.power(2.0, rel) - 1.0
    return float(np.sum(gains / discounts))


def _cv_ranking_metrics_grouped(
    y_true: np.ndarray,
    y_score: np.ndarray,
    q_groups: np.ndarray,
    k: int = CV_TOP_K,
    relevance_threshold: float = CV_RELEVANCE_THRESHOLD,
) -> tuple[float, float, float]:
    """Return mean NDCG, MRR, and HIT over grouped query rankings."""
    fold_df = pd.DataFrame(
        {
            "query": np.asarray(q_groups, dtype=str),
            "label": np.asarray(y_true, dtype=float),
            "score": np.asarray(y_score, dtype=float),
        }
    )

    ndcg_vals = []
    mrr_vals = []
    hit_vals = []

    for _, grp in fold_df.groupby("query", sort=False):
        ranked = grp.sort_values("score", ascending=False)
        relevances = ranked["label"].astype(float).tolist()

        ideal = _cv_dcg_at_k(sorted(relevances, reverse=True), k)
        ndcg_vals.append(0.0 if ideal == 0.0 else _cv_dcg_at_k(relevances, k) / ideal)

        binary_rel = (ranked["label"].astype(float) >= relevance_threshold).tolist()
        hit_vals.append(1.0 if any(binary_rel[:k]) else 0.0)

        mrr = 0.0
        for rank, is_rel in enumerate(binary_rel[:k], start=1):
            if is_rel:
                mrr = 1.0 / rank
                break
        mrr_vals.append(mrr)

    ndcg_mean = float(np.mean(ndcg_vals)) if ndcg_vals else 0.0
    mrr_mean = float(np.mean(mrr_vals)) if mrr_vals else 0.0
    hit_mean = float(np.mean(hit_vals)) if hit_vals else 0.0
    return ndcg_mean, mrr_mean, hit_mean


# ------------------------
# Grouped cross-validation on train split (NDCG-based)
# ------------------------

n_splits = 300  # min(5, len(set(g_train)))
print(f"Using {n_splits}-fold GroupKFold CV on train set for alpha tuning (NDCG@{CV_TOP_K})")
cv_results = []
best_model = None
if n_splits >= 2:
    gkf = GroupKFold(n_splits=n_splits)
    best_ndcg = -1.0
    for alpha in ALPHA_GRID:
        fold_ndcg = []
        fold_mrr = []
        fold_hit = []

        for fold_train_idx, fold_valid_idx in gkf.split(X_train, y_train, groups=g_train):
            fold_model = Pipeline(
                steps=[
                    # ("scaler", StandardScaler()),
                    ("ridge", Ridge(alpha=alpha, random_state=42)),
                ]
            )
                
            fold_model.fit(X_train.iloc[fold_train_idx], y_train[fold_train_idx])
            fold_pred = np.asarray(fold_model.predict(X_train.iloc[fold_valid_idx]), dtype=float).reshape(-1)

            fold_ndcg_score, fold_mrr_score, fold_hit_score = _cv_ranking_metrics_grouped(
                y_true=y_train[fold_valid_idx],
                y_score=fold_pred,
                q_groups=g_train[fold_valid_idx],
                k=CV_TOP_K,
                relevance_threshold=CV_RELEVANCE_THRESHOLD,
            )
            fold_ndcg.append(fold_ndcg_score)
            fold_mrr.append(fold_mrr_score)
            fold_hit.append(fold_hit_score)

            if fold_ndcg_score > best_ndcg:
                best_ndcg = fold_ndcg_score
                best_model = fold_model

        cv_results.append(
            {
                "alpha": alpha,
                "cv_ndcg_mean": float(np.mean(fold_ndcg)),
                "cv_ndcg_std": float(np.std(fold_ndcg)),
                "cv_mrr_mean": float(np.mean(fold_mrr)),
                "cv_mrr_std": float(np.std(fold_mrr)),
                "cv_hit_mean": float(np.mean(fold_hit)),
                "cv_hit_std": float(np.std(fold_hit)),
            }
        )

    cv_df = pd.DataFrame(cv_results).sort_values("cv_ndcg_mean", ascending=False)
    best_alpha = float(cv_df.iloc[0]["alpha"])
    print("Cross-validation results (NDCG/MRR/HIT):")
    print(cv_df.to_string(index=False))
else:
    best_alpha = 1.0
    cv_df = pd.DataFrame(
        [
            {
                "alpha": best_alpha,
                "cv_ndcg_mean": np.nan,
                "cv_ndcg_std": np.nan,
                "cv_mrr_mean": np.nan,
                "cv_mrr_std": np.nan,
                "cv_hit_mean": np.nan,
                "cv_hit_std": np.nan,
            }
        ]
    )
    print("Not enough train query groups for GroupKFold; fallback alpha=1.0")


# ------------------------
# Final model + predictions for ranking evaluation
# ------------------------
rating_model = Pipeline(
    steps=[
        # ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=best_alpha, random_state=42)),
    ]
)
rating_model = best_model if best_model is not None else rating_model.fit(X_train, y_train)
test_pred = np.asarray(rating_model.predict(X_test), dtype=float).reshape(-1)

print(f"Best alpha from CV (NDCG@{CV_TOP_K}): {best_alpha}")
print("Ranking metrics (NDCG/MRR/HIT) are computed in the next cell.")


# ------------------------
# Inference helper
# ------------------------
store_lookup_for_infer = store_full_df.drop_duplicates(subset=["RestaurantID"]).set_index("RestaurantID")


def predict_store_rating_from_query(user_query: str, store_id: int) -> dict:
    """Predict rating [0,4] for a (user_query, store_id) pair."""
    store_id = int(store_id)
    if store_id not in store_lookup_for_infer.index:
        return {
            "ok": False,
            "message": f"store_id={store_id} not found in store table",
        }

    store_row = store_lookup_for_infer.loc[store_id]
    if isinstance(store_row, pd.DataFrame):
        store_row = store_row.iloc[0]
    parsed_query = parse_user_all(str(user_query))
    x = pd.DataFrame([_build_pair_features(parsed_query, store_row)])

    pred = float(rating_model.predict(x)[0])
    pred_clipped = float(np.clip(pred, 0.0, 4.0))
    return {
        "ok": True,
        "prediction_continuous": pred,
        "prediction_clipped_0_4": pred_clipped,
        "prediction_rounded_0_4": int(np.rint(pred_clipped)),
    }


# Example inference
example = predict_store_rating_from_query(
    "Mình muốn tìm quán cơm gà ở Tân Bình có giao tận nơi và mở buổi trưa",
    769148,
 )
example

Rows: train=6375, test=1125
Unique queries: train=425, test=75
Using 300-fold GroupKFold CV on train set for alpha tuning (NDCG@5)
Cross-validation results (NDCG/MRR/HIT):
 alpha  cv_ndcg_mean  cv_ndcg_std  cv_mrr_mean  cv_mrr_std  cv_hit_mean  cv_hit_std
  5.00      0.777272     0.220711     0.663583    0.358041     0.796667    0.351647
  2.00      0.776298     0.220416     0.661639    0.357404     0.796667    0.351647
  0.50      0.776250     0.220287     0.661639    0.357404     0.796667    0.351647
  1.00      0.776196     0.220319     0.661639    0.357404     0.796667    0.351647
  0.01      0.776172     0.220420     0.661472    0.357607     0.796667    0.351647
  0.10      0.776149     0.220443     0.661472    0.357607     0.796667    0.351647
 10.00      0.775188     0.220174     0.658028    0.357173     0.796667    0.351647
Best alpha from CV (NDCG@5): 5.0
Ranking metrics (NDCG/MRR/HIT) are computed in the next cell.


{'ok': True,
 'prediction_continuous': 3.1265245688168877,
 'prediction_clipped_0_4': 3.1265245688168877,
 'prediction_rounded_0_4': 3}

### Testing model
Đánh giá kết quả theo NDCG, MRR, HIT ở mức top-k theo từng query group.

In [14]:
# Ranking-based evaluation (grouped by query): NDCG, MRR, HIT
TOP_K = 5
RELEVANCE_THRESHOLD = 3.0


def _dcg_at_k(relevances: list[float], k: int) -> float:
    rel = np.asarray(relevances[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rel.size + 2))
    gains = np.power(2.0, rel) - 1.0
    return float(np.sum(gains / discounts))


def _ndcg_at_k(relevances: list[float], k: int) -> float:
    ideal = _dcg_at_k(sorted(relevances, reverse=True), k)
    if ideal == 0.0:
        return 0.0
    return _dcg_at_k(relevances, k) / ideal


def _compute_ranking_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    query_groups: np.ndarray,
    top_k: int = TOP_K,
    relevance_threshold: float = RELEVANCE_THRESHOLD,
    prefix: str = "",
) -> tuple[dict, pd.DataFrame]:
    df = pd.DataFrame(
        {
            "query": np.asarray(query_groups, dtype=str),
            "label": np.asarray(y_true, dtype=float),
            "score": np.asarray(y_score, dtype=float),
        }
    )

    rows = []
    for query, grp in df.groupby("query", sort=False):
        ranked = grp.sort_values("score", ascending=False)
        relevances = ranked["label"].astype(float).tolist()
        binary_relevance = (ranked["label"].astype(float) >= relevance_threshold).tolist()

        ndcg = _ndcg_at_k(relevances, top_k)
        hit = 1.0 if any(binary_relevance[:top_k]) else 0.0

        mrr = 0.0
        for rank, is_rel in enumerate(binary_relevance[:top_k], start=1):
            if is_rel:
                mrr = 1.0 / rank
                break

        # rows.append({"query": query, "ndcg": ndcg, "mrr": mrr, "hit": hit, "label-score": list(zip(ranked["label"].tolist(), ranked["score"].tolist()))})
        rows.append({"query": query, "ndcg": ndcg, "mrr": mrr, "hit": hit})

    metrics_df = pd.DataFrame(rows)
    summary = {
        f"{prefix}NDCG@{top_k}": float(metrics_df["ndcg"].mean()) if len(metrics_df) else 0.0,
        f"{prefix}MRR@{top_k}": float(metrics_df["mrr"].mean()) if len(metrics_df) else 0.0,
        f"{prefix}HIT@{top_k}": float(metrics_df["hit"].mean()) if len(metrics_df) else 0.0,
        f"{prefix}num_queries": int(metrics_df["query"].nunique()) if len(metrics_df) else 0,
    }
    return summary, metrics_df


test_rank_summary, test_rank_df = _compute_ranking_metrics(
    y_true=y_test,
    y_score=test_pred,
    query_groups=g_test,
    top_k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
    prefix="Test_",
)

print("Test ranking metrics:")
for metric_name, metric_value in test_rank_summary.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value}")

print("\nSample of per-query ranking metrics:")
print(test_rank_df.head(10).to_string(index=False))

Test ranking metrics:
Test_NDCG@5: 0.7721
Test_MRR@5: 0.7189
Test_HIT@5: 0.8133
Test_num_queries: 75

Sample of per-query ranking metrics:
                                                                                                                                  query     ndcg      mrr  hit
                                                       Hãy cho tôi biết những quán cafe được đánh giá cao ở quận 3 để đi chiều Chủ nhật 0.568280 0.500000  1.0
                                                            Hãy cho tôi biết những quán ăn vặt ở huyện Bình Chánh phù hợp với sinh viên 0.876413 1.000000  1.0
                                               Hãy cho tôi tìm kiếm một quán ẩm thực Thái Lan có lượt xem nhiều và có được đánh giá cao 0.871726 1.000000  1.0
                          Hãy tìm kiếm giúp tôi một nhà hàng Ý ở quận 1 dành cho cặp đôi cho tối cuối tuần, có vị trí được đánh giá cao 0.906508 1.000000  1.0
                                                      Tôi muốn tìm

### Coefficient of model

In [15]:
pd.DataFrame([rating_model[-1].coef_], columns=X_train.columns.tolist())

,sim_location,sim_cuisine,sim_target_audience,sim_category,sim_service,sim_rating,sim_meta,fit_time,fit_price
0,2.823139,0.310718,0.0,0.212905,-0.330783,0.307561,-0.334058,-0.356265,-0.05841
